# Toucan Tool-Use Data Generation with ShopInsights MCP Server

This notebook demonstrates the end-to-end **Toucan pipeline** ([arXiv 2510.01179](https://arxiv.org/abs/2510.01179)) for generating high-quality tool-use training data using SDG Hub.

## Architecture

```
                        Toucan Pipeline (SDG Hub Flow)
┌──────────────────────────────────────────────────────────────────────────┐
│                                                                        │
│   ┌─────────┐    ┌──────────┐    ┌──────────┐    ┌─────────────────┐   │
│   │ Diversi-│    │  Task    │    │ Quality  │    │  Trajectory     │   │
│   │   ty    │───▶│Synthesis │───▶│ Filter   │───▶│  Generation     │   │
│   │         │    │(GPT-5.2) │    │(GPT-5.2) │    │  (Langflow)     │   │
│   └─────────┘    └──────────┘    └──────────┘    └────────┬────────┘   │
│                                                           │            │
│                                                           ▼            │
│                                                  ┌─────────────────┐   │
│                                                  │   Response      │   │
│                                                  │   Quality       │   │
│                                                  │   Validation    │   │
│                                                  │   (GPT-5.2)     │   │
│                                                  └─────────────────┘   │
└──────────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
                         Filtered training data
                          (question, trajectory,
                           quality scores)
```

## Components

| Component | Role |
|---|---|
| **ShopInsights MCP Server** | 15-tool e-commerce analytics platform (this repo) |
| **GPT-5.2** (teacher) | Generates questions from tool schemas + scores quality |
| **Langflow + Qwen3** (student) | Executes questions against MCP tools; produces trajectories |
| **SDG Hub Toucan Flow** | Orchestrates the 5-stage pipeline |

## Prerequisites

Before running this notebook:

1. **MCP Server** is running: `uv run python examples/agentic/ecommerce_mcp/server.py`
2. **Langflow** is running with a Qwen3 agent connected to the MCP server
3. You have an **OpenAI API key** for GPT-5.2

---
## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from pprint import pprint
import json
import os
import sys

import datasets

# Required to run the flow with async mode
import nest_asyncio
import pandas as pd

nest_asyncio.apply()

# # Ensure local imports work
# NOTEBOOK_DIR = Path.cwd()
# if str(NOTEBOOK_DIR) not in sys.path:
#     sys.path.insert(0, str(NOTEBOOK_DIR))

# # Project root (3 levels up from examples/agentic/ecommerce_mcp/)
# PROJECT_ROOT = NOTEBOOK_DIR.parent.parent.parent

# print(f"Notebook dir:  {NOTEBOOK_DIR}")
# print(f"Project root:  {PROJECT_ROOT}")

/Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ── Configuration ──────────────────────────────────────────────────────
# Fill in your credentials here or set them as environment variables.

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-...")
TEACHER_MODEL = "openai/gpt-5.2"  # LiteLLM model string

LANGFLOW_URL = "http://localhost:3000/api/v1/run/1513d1b0-c11a-4fed-9bf1-f34c0380e428"  # Replace with your Langflow flow URL
LANGFLOW_API_KEY = os.environ.get("LANGFLOW_API_KEY", None)

---
## 1. Explore the MCP Server

The **ShopInsights Analytics Platform** provides 15 tools organized into deliberate ambiguity clusters that make tool selection challenging for the student model:

| Cluster | Tools | Challenge |
|---|---|---|
| Product Discovery | `search_products`, `browse_catalog`, `get_trending_products`, `get_product_details` | Which search/browse tool is right? |
| Sales & Revenue | `get_sales_data`, `get_revenue_report`, `get_store_overview` | Per-product vs. aggregate vs. snapshot |
| Customer Analytics | `get_customer_segments`, `get_customer_profile`, `get_abandoned_carts` | Individual vs. aggregate vs. behavioral |
| Multi-step | `analyze_product_performance`, `compare_products`, `forecast_demand`, `get_inventory_status`, `create_promotion` | Require chaining 2-4 tools |

In [5]:
from data import create_data_store

store = create_data_store()

print(f"Products:        {len(store.products)}")
print(f"Customers:       {len(store.customers)}")
print(f"Orders:          {len(store.orders)}")
print(f"Inventory rows:  {len(store.inventory)}")
print(f"Abandoned carts: {len(store.abandoned_carts)}")
print(f"Promotions:      {len(store.promotions)}")

Products:        51
Customers:       30
Orders:          200
Inventory rows:  153
Abandoned carts: 15
Promotions:      5


In [6]:
# Show the category hierarchy and sample products
categories = sorted({p["category"] for p in store.products})
print("Category hierarchy:")
for cat in categories:
    count = sum(1 for p in store.products if p["category"] == cat)
    print(f"  {cat} ({count} products)")

print(f"\nSample product:")
pprint(store.products[0])

Category hierarchy:
  Clothing > Men (6 products)
  Clothing > Women (6 products)
  Electronics > Accessories (6 products)
  Electronics > Laptops (6 products)
  Electronics > Phones (5 products)
  Home & Kitchen > Appliances (6 products)
  Home & Kitchen > Furniture (5 products)
  Sports & Outdoors > Camping (5 products)
  Sports & Outdoors > Fitness (6 products)

Sample product:
{'avg_rating': 3.8,
 'brand': 'NovaPhone',
 'category': 'Electronics > Phones',
 'cost': 96.33,
 'created_at': '2024-10-11',
 'id': 'PROD-0001',
 'name': 'NovaPhone Flip Z',
 'price': 231.78,
 'review_count': 924,
 'specs': {'color': 'Red', 'weight_kg': 10.77},
 'tags': ['5G', 'waterproof', 'fast-charge']}


In [7]:
# Inspect all 15 tool schemas from the MCP server
from server import mcp

tools = await mcp.list_tools()

print(f"Total tools: {len(tools)}\n")
print(f"{'#':<3} {'Tool Name':<32} {'Params':>6}  Description")
print("─" * 100)
for i, t in enumerate(tools, 1):
    mt = t.to_mcp_tool()
    n_params = len(mt.inputSchema.get("properties", {}))
    desc = (mt.description or "").split("\n")[0][:50]
    print(f"{i:<3} {mt.name:<32} {n_params:>6}  {desc}")

Total tools: 15

#   Tool Name                        Params  Description
────────────────────────────────────────────────────────────────────────────────────────────────────
1   search_products                       8  Search the product catalog using keywords and filt
2   browse_catalog                        3  Browse products by category hierarchy.
3   get_trending_products                 4  Get products with rising performance metrics in a 
4   get_product_details                   1  Get full details for a single product by its ID.
5   get_sales_data                        4  Get unit sales data for specific products over a d
6   get_revenue_report                    3  Get aggregated revenue breakdown by a business dim
7   get_store_overview                    0  Get a quick dashboard snapshot of overall store pe
8   get_customer_segments                 1  Get aggregate customer segment breakdown.
9   get_customer_profile                  2  Get a single customer's full profil

In [8]:
# Deep-dive: inspect a complex tool schema (create_promotion has 9 params)
promo_tool = next(t for t in tools if t.name == "create_promotion")
schema = promo_tool.to_mcp_tool().inputSchema
print("create_promotion input schema:")
print(json.dumps(schema, indent=2))

create_promotion input schema:
{
  "additionalProperties": false,
  "properties": {
    "name": {
      "description": "Promotion name",
      "type": "string"
    },
    "product_ids": {
      "description": "Product IDs to include in promotion",
      "items": {
        "type": "string"
      },
      "type": "array"
    },
    "discount_type": {
      "description": "Type: percentage, fixed_amount, buy_one_get_one",
      "type": "string"
    },
    "discount_value": {
      "default": 0.0,
      "description": "Discount value (percentage or fixed amount; ignored for BOGO)",
      "type": "number"
    },
    "start_date": {
      "default": "",
      "description": "Start date (ISO format YYYY-MM-DD)",
      "type": "string"
    },
    "end_date": {
      "default": "",
      "description": "End date (ISO format YYYY-MM-DD)",
      "type": "string"
    },
    "min_quantity": {
      "anyOf": [
        {
          "type": "integer"
        },
        {
          "type": "null"
      

In [9]:
# Quick smoke test: call a few tools directly to verify the server logic
from server import search_products, get_store_overview, get_trending_products

print("=== Store Overview ===")
overview = get_store_overview()
pprint(overview)

print("\n=== Search: 'wireless' ===")
results = search_products(query="wireless", limit=3)
for p in results["products"]:
    print(f"  {p['id']}: {p['name']} — ${p['price']:.2f} ({p['category']})")

print(f"\n=== Trending (by revenue, last 90 days) ===")
trending = get_trending_products(metric="revenue", days=90, limit=5)
for p in trending["trending"]:
    print(f"  {p['id']}: {p['name']} — trend_score={p['trend_score']}")

=== Store Overview ===
{'average_order_value': 2032.95,
 'completed_orders': 33,
 'conversion_rate_pct': 16.5,
 'top_5_products': [{'name': 'AeroBook Gaming X17',
                     'product_id': 'PROD-0006',
                     'revenue': 58562.73},
                    {'name': 'CompuMax Chromebook Air',
                     'product_id': 'PROD-0008',
                     'revenue': 47973.03},
                    {'name': 'ZenMobile Edge Plus',
                     'product_id': 'PROD-0005',
                     'revenue': 38482.2},
                    {'name': 'AeroBook Ultrabook 14',
                     'product_id': 'PROD-0010',
                     'revenue': 36573.9},
                    {'name': 'NovaPhone Lite SE',
                     'product_id': 'PROD-0004',
                     'revenue': 20310.88}],
 'total_customers': 30,
 'total_orders': 200,
 'total_products': 51,
 'total_revenue': 406590.21}

=== Search: 'wireless' ===
  PROD-0012: SnapGear Screen Protector 2-Pack

---
## 2. Create the Input Dataset

The Toucan flow expects a HuggingFace `Dataset` or a pandas `DataFrame` with three columns:

| Column | Type | Description |
|---|---|---|
| `tool_list` | `list[dict]` | Tool dicts with `name`, `description`, `inputSchema` |
| `mcp_server_name` | `str` | Server name |
| `mcp_server_description` | `str` | Server description |

Each row represents one MCP server's tool collection. The flow then multiplies rows and samples tool subsets for diversity.

In [10]:
# Build tool_list from MCP server schemas
tool_list = []
for t in tools:
    mt = t.to_mcp_tool()
    tool_list.append({
        "name": mt.name,
        "description": mt.description or "",
        "inputSchema": mt.inputSchema,
    })

server_name = mcp.name or "ShopInsights Analytics Platform"
server_description = (
    "E-commerce analytics platform for an online retailer. "
    "Provides product search, sales analytics, customer insights, "
    "demand forecasting, and promotional management. "
    "Features 15 tools organized across product discovery, sales & revenue, "
    "customer analytics, and multi-step analytical workflows."
)

print(f"Extracted {len(tool_list)} tool schemas")
print(f"Server: {server_name}")

Extracted 15 tool schemas
Server: ShopInsights Analytics Platform


In [11]:
# Create and save the HuggingFace Dataset
ds = datasets.Dataset.from_dict({
    "tool_list": [tool_list],
    "mcp_server_name": [server_name],
    "mcp_server_description": [server_description],
})

---
## 3. Load and Inspect the Toucan Flow

The Toucan flow is a 5-stage pipeline defined in YAML with 17 blocks:

| Stage | Blocks | Purpose |
|---|---|---|
| 1. Diversity | `RowMultiplierBlock`, `SamplerBlock` | Multiply rows × 10, sample 2-tool subsets |
| 2. Task Synthesis | `PromptBuilderBlock`, `LLMChatBlock`, `TagParserBlock` | GPT-5.2 generates realistic multi-tool questions |
| 3. Quality Filter | `PromptBuilderBlock`, `LLMChatBlock`, `TagParserBlock`, `ColumnValueFilterBlock` | Score questions on 6 dimensions, keep "good"/"excellent" |
| 4. Trajectory Gen | `AgentBlock`, `AgentResponseExtractorBlock` | Langflow+Qwen3 executes questions against MCP server |
| 5. Response QA | `PromptBuilderBlock`, `LLMChatBlock`, `TagParserBlock`, `ColumnValueFilterBlock` × 2 | Score completeness + conciseness, filter low-quality |

In [12]:
from sdg_hub import Flow, FlowRegistry

FlowRegistry.discover_flows()

[23:29:20] INFO     Discovered 12 flows                                                             ]8;id=956423;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/registry.py\registry.py]8;;\:]8;id=652165;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/registry.py#126\126]8;;\

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID                  ┃ Name                 ┃ Author               ┃ Tags                 ┃ Description          ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ clean-shadow-397    │ Advanced Japanese    │ SDG Hub Contributors │ question-generation, │ A comprehensive flow │
│                     │ Document Grounded    │                      │ knowledge-extractio… │ that generates       │
│                     │ Question-Answer      │                      │ qa-pairs,            │ high-quality         │
│                     │ Generation Flow for  │                      │ document-processing, │ question-answer      │
│                     │ Knowledge Tuning     │                      │ educational,         │ pairs from Japanese  │
│                     │                      │                      │ multilingual,        │ input documents      │
│                     │                      │                      │ japanese             │ using multiple LLM   │
│                     │                      │                      │                      │ blocks for question  │
│                     │                      │                      │                      │ generation, answer   │
│                     │                      │                      │                      │ synthesis, and       │
│                     │                      │                      │                      │ quality evaluation.  │
│ epic-jade-656       │ Extractive Summary   │ SDG Hub Contributors │ knowledge-tuning,    │ Generate extractive  │
│                     │ Knowledge Tuning     │                      │ document-internaliz… │ summary from the     │
│                     │ Dataset Generation   │                      │ question-generation, │ input document. Each │
│                     │ Flow                 │                      │ knowledge-extractiv… │ document is first    │
│                     │                      │                      │ qa-pairs,            │ converted into list  │
│                     │                      │                      │ extractive-summaries │ of knowledge         │
│                     │                      │                      │                      │ segments for         │
│                     │                      │                      │                      │ creating extractive  │
│                     │                      │                      │                      │ summary and then     │
│                     │                      │                      │                      │ annotated with       │
│                     │                      │                      │                      │ context,             │
│                     │                      │                      │                      │ relationship and     │
│                     │                      │                      │                      │ relevance. This is   │
│                     │                      │                      │                      │ then converted into  │
│                     │                      │                      │                      │ Question-Answer      │
│                     │                      │                      │                      │ pairs.               │
│ epic-jade-656-es    │ Extractive Summary   │ SDG Hub Contributors │ knowledge-tuning,    │ Generate extractive  │
│                     │ Knowledge Tuning     │                      │ document-internaliz… │ summary from the     │
│                     │ Dataset Generation   │                      │ question-generation, │ input document in    │
│                     │ Flow (Spanish)       │                      │ knowledge-extractiv… │ Spanish. Each        │
│                     │                      │          

In [13]:
# Load the Toucan flow
flow_id = "smart-pine-839"
flow_path = FlowRegistry.get_flow_path(flow_id)
flow = Flow.from_yaml(flow_path)

# Print the full flow pipeline structure
flow.print_info()

[23:29:20] INFO     Loading flow from:                                                          ]8;id=448161;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/serialization.py\serialization.py]8;;\:]8;id=171286;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/serialization.py#61\61]8;;\
                    /Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/flows/a                    
                    gentic/tool_datagen/flow.yaml                                                                  

╭─────────────────────────────────────────────── Flow Information ────────────────────────────────────────────────╮
│ Toucan Tool-Use Data Generation Flow                                                                            │
│ ├── Metadata                                                                                                    │
│ │   ├── Version: 1.0.0                                                                                          │
│ │   ├── Author: Xu et al. (2025) & SDG Hub Contributors                                                         │
│ │   └── Description: Generates high-quality tool-use training data from MCP tool collections using the Toucan   │
│ │       pipeline (arxiv 2510.01179). The pipeline synthesizes realistic user questions from tool descriptions,  │
│ │       filters for quality, generates agent execution trajectories, and validates response quality. See        │
│ │       https://arxiv.org/abs/2510.01179 for more details.                                                      │
│ │                                                                                                               │
│ └── Blocks (19 total)                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Block Details ─────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓ │
│ ┃ Block Name                ┃ Type                     ┃ Input Cols                ┃ Output Cols              ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩ │
│ │ multiply_tool_rows        │ RowMultiplierBlock       │ None                      │ None                     │ │
│ │ sample_tools              │ SamplerBlock             │ ['tool_list']             │ ['sampled_tools']        │ │
│ │ build_question_gen_prompt │ PromptBuilderBlock       │ {'sampled_tools':         │ ['question_gen_messages… │ │
│ │                           │                          │ 'tool_list',              │                          │ │
│ │                           │                          │ 'mcp_server_name':        │                          │ │
│ │                           │                          │ 'mcp_server_name',        │                          │ │
│ │                           │                          │ 'mcp_server_description': │                          │ │
│ │                           │                          │ 'mcp_server_description'} │                          │ │
│ │ generate_questions        │ LLMChatBlock             │ ['question_gen_messages'] │ ['question_gen_response… │ │
│ │ extract_question_response │ LLMResponseExtractorBlo… │ ['question_gen_response'] │ ['question_gen_content'] │ │
│ │ parse_question_fields     │ TagParserBlock           │ ['question_gen_content']  │ ['target_tools',         │ │
│ │                           │                          │                           │ 'question']              │ │
│ │ build_quality_check_prom… │ PromptBuilderBlock       │ {'tool_list':             │ ['quality_check_message… │ │
│ │                           │                          │ 'all_server_and_tool_inf… │                          │ │
│ │                           │                          │ 'question':               │                          │ │
│ │                           │                          │ 'question_content',       │                          │ │
│ │                           │                          │ 'target_tools':           │                          │ │
│ │                           │                          │ 'intended_tool'}          │                          │ │
│ │ score_question_quality    │ LLMChatBlock             │ ['quality_check_messages… │ ['quality_check_respons… │ │
│ │ extract_quality_response  │ LLMResponseExtractorBlo… │ ['quality_check_response… │ ['quality_check_content… │ │
│ │ parse_quality_score       │ TagParserBlock           │ ['quality_check_content'] │ ['question_quality_rati… │ │
│ │ filter_low_quality_quest… │ ColumnValueFilterBlock   │ ['question_quality_ratin… │ None                     │ │
│ │ run_agent_trajectory      │ AgentBlock               │ ['question']              │ ['agent_response']       │ │
│ │ extract_agent_text        │ AgentResponseExtractorB… │ ['agent_response']        │ ['extract_agent_text_te… │ │
│ │ build_response_quality_p… │ PromptBuilderBlock       │ {'question':              │ ['response_quality_mess… │ │
│ │                           │                          │ 'question_content',       │                          │ │
│ │                           │                          │ 'target_tools':           │                          │ │
│ │                           │                          │ 'intended_tool',          │                          │ │
│ │                           │                          │ 'extract_agent_text_text… │                          │ │
│ │                           │                          │ 'conversation_history'}   │                          │ │
│ │ score_response_quality    │ LLMChatBlock            

---
## 4. Configure the Pipeline

The flow needs two runtime configurations:

1. **Teacher model** (GPT-5.2) — used by all `LLMChatBlock`s for question generation and quality scoring
2. **Langflow agent** — used by the `AgentBlock` to execute questions through the Qwen3 agent
3. **Num of samples** — used by the `RowMultiplierBlock` to multiply the number of rows by the specified factor

In [14]:
# Check recommended models
print("Default model:", flow.get_default_model())
print("Model recommendations:", flow.get_model_recommendations())

Default model: openai/gpt-5.2
Model recommendations: {'default': 'openai/gpt-5.2', 'compatible': ['openai/gpt-5-mini', 'hosted_vllm/openai/gpt-oss-120b'], 'experimental': []}


In [ ]:
# Configure the teacher model (GPT-5.2 for question gen + quality scoring)
flow.set_model_config(
    model=TEACHER_MODEL,
    api_key=OPENAI_API_KEY,
)

[23:29:22] INFO     Auto-detected 3 LLM blocks for configuration: ['generate_questions',        ]8;id=300374;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=138010;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/model_config.py#242\242]8;;\
                    'score_question_quality', 'score_response_quality']                                            

           INFO     Successfully configured 3 LLM blocks with: model: 'openai/gpt-5.2',         ]8;id=954936;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=220088;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/model_config.py#300\300]8;;\
                    api_key: (redacted)                                                                            

           INFO     Configured blocks: ['generate_questions', 'score_question_quality',         ]8;id=347401;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=671136;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/model_config.py#303\303]8;;\
                    'score_response_quality']                                                                      

In [18]:
# Configure the Langflow agent (Qwen3 with MCP server)
agent_kwargs = {
    "agent_framework": "langflow",
    "agent_url": LANGFLOW_URL,
}
if LANGFLOW_API_KEY:
    agent_kwargs["agent_api_key"] = LANGFLOW_API_KEY

flow.set_agent_config(**agent_kwargs)

[23:29:39] INFO     Auto-detected 1 agent blocks for configuration: ['run_agent_trajectory']    ]8;id=913977;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/agent_config.py\agent_config.py]8;;\:]8;id=611116;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/agent_config.py#170\170]8;;\

           INFO     Successfully configured 1 agent blocks with: agent_framework: 'langflow',   ]8;id=787190;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/agent_config.py\agent_config.py]8;;\:]8;id=29887;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/agent_config.py#228\228]8;;\
                    agent_url:                                                                                     
                    'http://localhost:3000/api/v1/run/1513d1b0-c11a-4fed-9bf1-f34c0380e428',                       
                    agent_api_key: (redacted)                                                                      

           INFO     Configured blocks: ['run_agent_trajectory']                                 ]8;id=284681;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/agent_config.py\agent_config.py]8;;\:]8;id=280220;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/agent_config.py#232\232]8;;\

In [28]:
# Configure the number of samples
# For this example, we'll use 10 samples

flow.blocks[0].num_samples = 1

---
## 5. Run the Pipeline

The `flow.generate()` method executes all 17 blocks in sequence. With checkpointing enabled, you can resume from where you left off if the pipeline is interrupted.

**Expected data flow:**
```
1 row (15 tools) 
  → ×10 multiplier = 10 rows
  → sample 2 tools each = 10 rows with tool subsets
  → generate questions = 10 rows with questions
  → quality filter = ~6-8 rows ("good"/"excellent" only)
  → agent trajectories = ~6-8 rows with responses
  → response quality filter = ~4-6 final training examples
```

In [30]:
# Run the full pipeline
result = flow.generate(ds)

[23:46:14] INFO     Converting datasets.Dataset to pd.DataFrame for processing                      ]8;id=936809;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=737232;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/execution.py#98\98]8;;\

           INFO     Starting flow 'Toucan Tool-Use Data Generation' v1.0.0 with 1 samples across   ]8;id=713474;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=327135;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/execution.py#458\458]8;;\
                    19 blocks                                                                                      

           INFO     Executing block 1/19: multiply_tool_rows (RowMultiplierBlock)                  ]8;id=605317;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=549457;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/execution.py#220\220]8;;\

╭────────────────────────────────────────────── multiply_tool_rows ───────────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: RowMultiplierBlock                                                                                  │
│ Input Rows: 1                                                                                                   │
│ Input Columns: 3                                                                                                │
│ Column Names: tool_list, mcp_server_name, mcp_server_description                                                │
│ Expected Output Columns: None specified                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── multiply_tool_rows - Complete ─────────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 1 → 1                                                                                                     │
│ Columns: 3 → 3                                                                                                  │
│ 📋 Final Columns: mcp_server_description, mcp_server_name, tool_list                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

           INFO     Executing block 18/19: filter_incomplete_responses (ColumnValueFilterBlock)    ]8;id=438433;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=268270;file:///Users/shiv/workspace/sdg_hub-feat-agentic-tool-datagen/src/sdg_hub/core/flow/execution.py#220\220]8;;\

╭────────────────────────────────────────── filter_incomplete_responses ──────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: ColumnValueFilterBlock                                                                              │
│ Input Rows: 1                                                                                                   │
│ Input Columns: 20                                                                                               │
│ Column Names: tool_list, mcp_server_name, mcp_server_description, sampled_tools, question_gen_messages,         │
│ question_gen_response, question_gen_content, target_tools, question, quality_check_messages,                    │
│ quality_check_response, quality_check_content, question_quality_rating, agent_response,                         │
│ extract_agent_text_text, response_quality_messages, response_quality_response, response_quality_content,        │
│ completeness_rating, conciseness_rating                                                                         │
│ Expected Output Columns: None specified                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────── filter_incomplete_responses - Complete ─────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 1 → 0                                                                                                     │
│ Columns: 20 → 20                                                                                                │
│ 📋 Final Columns: agent_response, completeness_rating, conciseness_rating, extract_agent_text_text,             │
│ mcp_server_description, mcp_server_name, quality_check_content, quality_check_messages, quality_check_response, │
│ question, question_gen_content, question_gen_messages, question_gen_response, question_quality_rating,          │
│ response_quality_content, response_quality_messages, response_quality_response, sampled_tools, target_tools,    │
│ tool_list                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────── Toucan Tool-Use Data Generation - Failed ────────────────────────────────────╮
│                                        Flow Execution Summary                                                   │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓           │
│ ┃ Block Name           ┃ Type            ┃   Duration ┃     Rows     ┃     Columns     ┃   Status   ┃           │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩           │
│ │ multiply_tool_rows   │ RowMultiplierB… │      0.00s │    1 → 1     │        —        │     ✓      │           │
│ │ sample_tools         │ SamplerBlock    │      0.00s │    1 → 1     │       +1        │     ✓      │           │
│ │ build_question_gen_… │ PromptBuilderB… │      0.00s │    1 → 1     │       +1        │     ✓      │           │
│ │ generate_questions   │ LLMChatBlock    │      4.49s │    1 → 1     │       +1        │     ✓      │           │
│ │ extract_question_re… │ LLMResponseExt… │      0.00s │    1 → 1     │       +1        │     ✓      │           │
│ │ parse_question_fiel… │ TagParserBlock  │      0.00s │    1 → 1     │       +2        │     ✓      │           │
│ │ build_quality_check… │ PromptBuilderB… │      0.01s │    1 → 1     │       +1        │     ✓      │           │
│ │ score_question_qual… │ LLMChatBlock    │     47.07s │    1 → 1     │       +1        │     ✓      │           │
│ │ extract_quality_res… │ LLMResponseExt… │      0.00s │    1 → 1     │       +1        │     ✓      │           │
│ │ parse_quality_score  │ TagParserBlock  │      0.00s │    1 → 1     │       +1        │     ✓      │           │
│ │ filter_low_quality_… │ ColumnValueFil… │      0.00s │    1 → 1     │        —        │     ✓      │           │
│ │ run_agent_trajectory │ AgentBlock      │     11.08s │    1 → 1     │       +1        │     ✓      │           │
│ │ extract_agent_text   │ AgentResponseE… │      0.00s │    1 → 1     │       +1        │     ✓      │           │
│ │ build_response_qual… │ PromptBuilderB… │      0.01s │    1 → 1     │       +1        │     ✓      │           │
│ │ score_response_qual… │ LLMChatBlock    │     14.30s │    1 → 1     │       +1        │     ✓      │           │
│ │ extract_response_qu… │ LLMResponseExt… │      0.00s │    1 → 1     │       +1        │     ✓      │           │
│ │ parse_response_scor… │ TagParserBlock  │      0.00s │    1 → 1     │       +2        │     ✓      │           │
│ ├──────────────────────┼─────────────────┼────────────┼──────────────┼─────────────────┼────────────┤           │
│ │ TOTAL                │ 17 blocks       │     76.99s │   0 final    │     0 final     │   17/17    │           │
│ └──────────────────────┴─────────────────┴────────────┴──────────────┴─────────────────┴────────────┘           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

EmptyDatasetError: Block 'filter_incomplete_responses' received an empty dataset
Details: Dataset must contain at least one sample for processing

In [ ]:
print(f"Pipeline complete!")
print(f"Generated {len(result)} training examples")
print(f"Columns: {result.column_names if hasattr(result, 'column_names') else list(result.columns)}")

---
## 6. Analyze Results

Let's inspect the generated training data — each row contains a realistic question, the agent's execution trajectory, and quality scores.

In [ ]:
# Convert to DataFrame for easier exploration
if hasattr(result, "to_pandas"):
    df = result.to_pandas()
else:
    df = result

print(f"Shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
for col in df.columns:
    dtype = df[col].dtype
    print(f"  {col}: {dtype}")

In [ ]:
# Show the key columns for each training example
key_cols = [c for c in ["question", "target_tools", "question_quality_rating",
                         "completeness_rating", "conciseness_rating"] if c in df.columns]

if key_cols:
    display(df[key_cols])
else:
    print("Key columns not found — check column names above")

In [ ]:
# Deep dive into a single training example
if len(df) > 0:
    row = df.iloc[0]

    print("=" * 80)
    print("SAMPLE TRAINING EXAMPLE")
    print("=" * 80)

    if "question" in row:
        print(f"\n--- Question ---")
        print(row["question"])

    if "target_tools" in row:
        print(f"\n--- Target Tools ---")
        print(row["target_tools"])

    if "question_quality_rating" in row:
        print(f"\n--- Quality Rating ---")
        print(row["question_quality_rating"])

    # Show the agent trajectory (the main training signal)
    trajectory_col = next((c for c in df.columns if "text" in c.lower() or "trajectory" in c.lower() or "agent" in c.lower()), None)
    if trajectory_col:
        print(f"\n--- Agent Trajectory ({trajectory_col}) ---")
        text = str(row[trajectory_col])
        # Show first 2000 chars
        if len(text) > 2000:
            print(text[:2000])
            print(f"\n... ({len(text) - 2000} more characters)")
        else:
            print(text)

    if "completeness_rating" in row:
        print(f"\n--- Response Scores ---")
        print(f"  Completeness: {row.get('completeness_rating', 'N/A')}")
        print(f"  Conciseness:  {row.get('conciseness_rating', 'N/A')}")
else:
    print("No results generated — check pipeline logs above.")

In [ ]:
# Quality distribution
for col in ["question_quality_rating", "completeness_rating", "conciseness_rating"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts().to_string())

---
## 7. Export

In [ ]:
# Save as Parquet
output_path = NOTEBOOK_DIR / "toucan_ecommerce_results.parquet"
df.to_parquet(output_path, index=False)
print(f"Results saved to {output_path}")
print(f"  Rows: {len(df)}")
print(f"  Size: {output_path.stat().st_size / 1024:.1f} KB")

---

## Summary

This notebook demonstrated the complete Toucan pipeline:

1. **MCP Server** — 15 e-commerce analytics tools with deliberate ambiguity clusters
2. **Dataset Creation** — Extracted tool schemas into the format expected by the flow
3. **Flow Configuration** — Connected GPT-5.2 (teacher) and Langflow+Qwen3 (student)
4. **Pipeline Execution** — Ran the 5-stage Toucan pipeline with checkpointing
5. **Results Analysis** — Inspected generated training examples with quality scores

The output parquet file contains high-quality tool-use training data that can be used to fine-tune Qwen3 (or other models) on this specific MCP server's tool set.